## Basic RAG with Ollama + ChromaDB

This notebook demonstrates a basic Retrieval-Augmented Generation (RAG) pipeline
using local Ollama models and ChromaDB as the vector store.


In [1]:
import os
import sys

IN_COLAB  = "google.colab" in sys.modules
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or os.path.exists("/kaggle/input")

if IN_COLAB or IN_KAGGLE:
    !pip install git+https://github.com/saikrishna1729/reliablerag.git@rag_pipeline/jithu datasets pandas -q

In [2]:
%load_ext autoreload
%autoreload 2

In [2]:
import time

import pandas as pd
from datasets import load_dataset
from langchain_core.runnables import RunnableConfig
from langchain_text_splitters import RecursiveCharacterTextSplitter

from reliablerag.chain import build_rag_chain, TimingCallbackHandler
from reliablerag.env import load_secrets
from reliablerag.evaluation import evaluate
from reliablerag.providers import create_embeddings, create_llm
from reliablerag.retriever import build_vector_store, get_retriever

### 1. Configuration

Model names and paths are loaded from `.env`. Fallback defaults are used if not set.

In [3]:
load_secrets()

PROVIDER           = os.environ["PROVIDER"]
EMBEDDING_MODEL    = os.environ["EMBEDDING_MODEL"]
LLM_MODEL          = os.environ["LLM_MODEL"]
CHROMA_PERSIST_DIR = os.environ["CHROMA_PERSIST_DIR"]

print(f"Provider        : {PROVIDER}")
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"LLM model       : {LLM_MODEL}")
print(f"Chroma dir      : {CHROMA_PERSIST_DIR}")

Provider        : ollama
Embedding model : nomic-embed-text-v2-moe:latest
LLM model       : gemma4:12b-it-q4_K_M
Chroma dir      : /Users/jithamanyu.manne/git/others/python/reliablerag/data/chroma_db


In [4]:
embeddings = create_embeddings(PROVIDER, EMBEDDING_MODEL)
llm = create_llm(PROVIDER, LLM_MODEL)

# Separate LLM for the TRACe judge with deterministic decoding.
# Same model, but temperature=0 so repeat evals don't drift on the same inputs.
judge_llm = create_llm(PROVIDER, LLM_MODEL, temperature=0)

### 2. Load CUAD Samples from RAGBench

Each sample contains a legal contract (`documents`), a `question`, a reference `response` generated
by Claude 3 Haiku, and pre-computed **TRACe labels** annotated by GPT-4:
`adherence`, `context_relevance`, `utilization`, `completeness`.

In [5]:
N_SAMPLES = 5  # increase to evaluate more samples

dataset = load_dataset("galileo-ai/ragbench", "cuad", split="train")
samples = list(dataset.select(range(N_SAMPLES)))


def fmt(v):
    return f"{v:.3f}" if v is not None else "N/A"


print(f"Loaded {len(samples)} CUAD samples")

s = samples[0]
print(f"\nQuestion         : {s['question']}")
print(f"Doc length       : {len(s['documents'][0])} chars")
print(f"Adherence score  : {s['adherence_score']}")
print(f"Relevance score  : {fmt(s['relevance_score'])}")
print(f"Utilization score: {fmt(s['utilization_score'])}")
print(f"Completeness     : {fmt(s['completeness_score'])}")
print(f"\nAlso available   : ragas_faithfulness={fmt(s['ragas_faithfulness'])}, "
      f"trulens_groundedness={fmt(s['trulens_groundedness'])}")

Loaded 5 CUAD samples

Question         : Is one party required to deposit its source code into escrow with a third party, which can be released to the counterparty upon the occurrence of certain events (bankruptcy,  insolvency, etc.)?
Doc length       : 122054 chars
Adherence score  : True
Relevance score  : 0.000
Utilization score: 0.000
Completeness     : 1.000

Also available   : ragas_faithfulness=N/A, trulens_groundedness=N/A


In [6]:
df_preview = dataset.to_pandas()
# df_preview.head(10)

### 3. Run RAG on Each Sample

CUAD contracts are up to 11k tokens each, so we chunk each document before indexing.
We build a fresh ephemeral vector store per sample (CUAD has 1 doc per question, so no cross-contamination).

In [8]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

results = []

for i, sample in enumerate(samples):
    question = sample["question"]
    raw_doc  = sample["documents"][0]  # CUAD: always 1 doc per sample

    print(f"\n[{i+1}/{len(samples)}] {question[:90]}...")

    t0 = time.perf_counter()
    chunks = splitter.create_documents(
        [raw_doc],
        metadatas=[{"source": f"cuad_sample_{i}"}],
    )
    print(f"[timing] chunking : {time.perf_counter() - t0:.3f}s  ({len(chunks)} chunks)")

    t0 = time.perf_counter()
    vector_store = build_vector_store(chunks, embeddings)
    print(f"[timing] chrome.from_documents : {time.perf_counter() - t0:.3f}s")
    retriever = get_retriever(vector_store, 20)

    retrieved_chunks = retriever.invoke(question)

    rag_chain    = build_rag_chain(retriever, llm=llm)
    our_response = rag_chain.invoke(question, config=RunnableConfig(callbacks=[TimingCallbackHandler()]))

    print(f"  → {our_response[:100]}...")

    results.append({
        "question"            : question,
        "context"             : raw_doc,
        "retrieved_chunks"    : retrieved_chunks,
        "our_response"        : our_response,
        "ref_response"        : sample["response"],
        "ref_adherence"       : bool(sample["adherence_score"]),
        "ref_relevance"       : sample["relevance_score"],
        "ref_utilization"     : sample["utilization_score"],
        "ref_completeness"    : sample["completeness_score"],
        "ragas_faithfulness"  : sample["ragas_faithfulness"],
        "trulens_groundedness": sample["trulens_groundedness"],
    })

    vector_store.delete_collection()

print(f"\nDone — {len(results)} samples processed.")


[1/5] Is one party required to deposit its source code into escrow with a third party, which can...
[timing] chunking : 0.008s  (291 chunks)
[timing] chrome.from_documents : 4.389s
[timing] retriever: 0.109s
[timing] llm      : 30.364s
  → The provided context does not contain any mention of source code, escrow requirements, or the releas...

[2/5] Does the contract contain a license granted by one party to its counterparty?...
[timing] chunking : 0.001s  (47 chunks)
[timing] chrome.from_documents : 0.578s
[timing] retriever: 0.116s
[timing] llm      : 30.686s
  → Based on the provided context, there is no mention of a license granted by one party to its counterp...

[3/5] The date when the contract is effective ...
[timing] chunking : 0.005s  (168 chunks)
[timing] chrome.from_documents : 1.875s
[timing] retriever: 0.115s
[timing] llm      : 27.583s
  → The document does not provide a specific calendar date for when the contract becomes effective. Inst...

[4/5] Does the contract cont

In [10]:
# Inspect the top-k retrieved chunks for one sample.
# Set SAMPLE_IDX to 0..N_SAMPLES-1. Sample 1 (Q2) is the most diagnostic — ref_relevance=0.833 but ours=0.
SAMPLE_IDX = 1

r = results[SAMPLE_IDX]
print(f"Q: {r['question']}\n")
print(f"Retrieved {len(r['retrieved_chunks'])} chunks:\n")
for j, chunk in enumerate(r["retrieved_chunks"]):
    print(f"--- chunk {j} ---")
    print(chunk.page_content)
    print()

Q: Does the contract contain a license granted by one party to its counterparty?

Retrieved 20 chunks:

--- chunk 0 ---
Counterparts

4.11 This Agreement may be executed in counterparts, each of which when delivered (whether in originally executed form or by facsimile transmission) will be deemed to be an  original and all of which together will constitute one and the same document.

IN WITNESS WHEREOF this Agreement has been executed by the parties hereto on the day  and year first above written.

GLAMIS GOLD LTD.

Per: Authorized Signatory

WESTERN COPPER CORPORATION

Per: Authorized Signatory

1162967.3

--- chunk 1 ---
4.2 The delivery of any notice, direction or other instrument, or a copy thereof, to a  party hereunder will be deemed to constitute the representation and warranty of the party who  has delivered it to the other party that such delivering party' is authorized to deliver such notice,  direction or other instrument at such time under this Agreement (unless the receivi

### 4. TRACe Evaluation

We evaluate our generated responses using the TRACe framework (paper: RAGBench, Friel et al.).

| Metric | What | How |
|---|---|---|
| **Adherence** | Is our response fully grounded in context? | LLM-as-judge |
| **Relevance** | Fraction of context relevant to query | From dataset (GPT-4 annotated) |
| **Utilization** | Fraction of context used in response | From dataset (GPT-4 annotated) |
| **Completeness** | Fraction of relevant context captured | From dataset (GPT-4 annotated) |

Relevance/Utilization/Completeness are properties of the document and retrieval setup (fixed per sample),
so we use the dataset's pre-computed GPT-4 labels as reference. We compute **Adherence** ourselves
for our generated response using an LLM judge.

In [12]:
import json

# n_runs > 1 averages the judge over N calls to damp LLM-as-judge variance.
# Pair with a deterministic judge (temperature=0 above) for tightest signal.
JUDGE_N_RUNS = 3

print(f"Evaluating TRACe metrics for each response (n_runs={JUDGE_N_RUNS})...\n")
for r in results:
    scores = evaluate(judge_llm, r["question"], r["retrieved_chunks"], r["our_response"], n_runs=JUDGE_N_RUNS)

    r["our_adherence"]             = scores.adherence
    r["our_adherence_explanation"] = scores.adherence_explanation
    r["our_relevance"]             = scores.relevance
    r["our_relevance_explanation"] = scores.relevance_explanation
    r["our_utilization"]           = scores.utilization
    r["our_completeness"]          = scores.completeness
    r["our_annotation"]            = scores.annotation

    status = "PASS" if scores.adherence else "FAIL"
    print(f"[{status}] {r['question'][:70]}...")
    print(f"  Adherence   : {status}  — {scores.adherence_explanation[:90]}")
    print(f"  Relevance   : {scores.relevance:.3f} — {scores.relevance_explanation[:90]}")
    print(f"  Utilization : {scores.utilization:.3f}")
    print(f"  Completeness: {scores.completeness:.3f}")
    print()

Evaluating TRACe metrics for each response (n_runs=3)...

[PASS] Is one party required to deposit its source code into escrow with a th...
  Adherence   : PASS  — The response is supported because it correctly identifies that the provided text does not 
  Relevance   : 0.000 — None of the provided documents (0a through 19c) contain information regarding 'source code
  Utilization : 0.000
  Completeness: 0.000

[PASS] Does the contract contain a license granted by one party to its counte...
  Adherence   : PASS  — Sentence a is supported because there is no mention of a 'license' in any part of the prov
  Relevance   : 0.146 — Documents 6a, 6b, 13a, and 14a are relevant because they define the core terms of the agre
  Utilization : 0.146
  Completeness: 1.000

[PASS] The date when the contract is effective ...
  Adherence   : PASS  — Sentence 'a' claims that no specific calendar date is provided for when the contract becom
  Relevance   : 0.034 — Document 6a contains information regardi

### 5. Results Summary

In [13]:
rows = []
for r in results:
    rows.append({
        "Question"          : r["question"][:50] + "...",
        "Our Adh."          : "✓" if r["our_adherence"] else "✗",
        "Ref Adh."          : "✓" if r["ref_adherence"] else "✗",
        "Our Rel."          : fmt(r["our_relevance"]),
        "Ref Rel."          : fmt(r["ref_relevance"]),
        "Our Util."         : fmt(r["our_utilization"]),
        "Ref Util."         : fmt(r["ref_utilization"]),
        "Our Comp."         : fmt(r["our_completeness"]),
        "Ref Comp."         : fmt(r["ref_completeness"]),
        "RAGAS Faith."      : fmt(r["ragas_faithfulness"]),
        "TruLens Ground."   : fmt(r["trulens_groundedness"]),
    })

df = pd.DataFrame(rows)
print("=" * 100)
print("TRACe Evaluation — CUAD RAGBench")
print("=" * 100)
print(df.to_string(index=False))

our_adh_rate = sum(r["our_adherence"] for r in results) / len(results)
ref_adh_rate = sum(r["ref_adherence"] for r in results) / len(results)

print(f"\nOur Adherence Rate : {our_adh_rate:.0%}  ({sum(r['our_adherence'] for r in results)}/{len(results)} samples)")
print(f"Ref Adherence Rate : {ref_adh_rate:.0%}  ({sum(r['ref_adherence'] for r in results)}/{len(results)} samples)")

print(f"\nAvg Our Relevance   : {sum(r['our_relevance'] for r in results)/len(results):.3f}")
print(f"Avg Our Utilization : {sum(r['our_utilization'] for r in results)/len(results):.3f}")
print(f"Avg Our Completeness: {sum(r['our_completeness'] for r in results)/len(results):.3f}")

non_null = [r for r in results if r["ref_relevance"] is not None]
if non_null:
    print(f"\nAvg Ref Relevance   : {sum(r['ref_relevance'] for r in non_null)/len(non_null):.3f}")
    print(f"Avg Ref Utilization : {sum(r['ref_utilization'] for r in non_null)/len(non_null):.3f}")
    print(f"Avg Ref Completeness: {sum(r['ref_completeness'] for r in non_null)/len(non_null):.3f}")

print("\n--- Per-sample responses ---")
for i, r in enumerate(results):
    print(f"\n[{i+1}] Q: {r['question']}")
    print(f"     Our : {r['our_response']}")
    print(f"     Ref : {r['ref_response']}")

TRACe Evaluation — CUAD RAGBench
                                             Question Our Adh. Ref Adh. Our Rel. Ref Rel. Our Util. Ref Util. Our Comp. Ref Comp. RAGAS Faith. TruLens Ground.
Is one party required to deposit its source code i...        ✓        ✓    0.000    0.000     0.000     0.000     0.000     1.000          N/A             N/A
Does the contract contain a license granted by one...        ✓        ✓    0.146    0.833     0.146     0.036     1.000     0.043          N/A             N/A
          The date when the contract is effective ...        ✓        ✓    0.034    0.003     0.034     0.003     1.000     1.000          N/A             N/A
Does the contract contain a clause that would awar...        ✓        ✗    0.225    0.038     0.112     0.003     0.500     0.091          N/A             N/A
                          The date of the contract...        ✓        ✓    0.028    0.004     0.028     0.002     1.000     0.500          N/A             N/A

Our Adherenc